In [ ]:
import "dotenv/config";
import {
    loadDocumentsFromDataDir,
    splitDocuments,
    initializeVectorstoreWithDocuments,
    createDocumentRetrievalChain,
    createRephraseQuestionChain,
    createHRAnswerPrompt,
  } from "./lib/helpers.ts";
  
  import {
    RunnablePassthrough,
    RunnableSequence,
    RunnableWithMessageHistory,
  } from "@langchain/core/runnables";
  import { ChatOpenAI } from "@langchain/openai";
  import { HttpResponseOutputParser } from "langchain/output_parsers";
  import { ChatMessageHistory } from "langchain/stores/message/in_memory";
  
  /* -------------------- Step 1: Prepare documents -------------------- */
  console.log("Loading documents…");
  const docs = await loadDocumentsFromDataDir();
  const splits = await splitDocuments(docs, 1536, 128);
  const vectorstore = await initializeVectorstoreWithDocuments({ documents: splits });
  const retriever = vectorstore.asRetriever();
  
  /* -------------------- Step 2: Build chains -------------------- */
  const rephraseQuestionChain = createRephraseQuestionChain();
  const docRetrievalChain = createDocumentRetrievalChain(retriever);
  const answerPrompt = createHRAnswerPrompt();
  
  const conversationalRetrievalChain = RunnableSequence.from([
    RunnablePassthrough.assign({ standalone_question: rephraseQuestionChain }),
    RunnablePassthrough.assign({ context: docRetrievalChain }),
    answerPrompt,
    new ChatOpenAI({ modelName: "gpt-3.5-turbo-1106", temperature: 0 }),
  ]);
  
  /* -------------------- Step 3: History + output parser -------------------- */
  const httpResponseOutputParser = new HttpResponseOutputParser({
    contentType: "application/json",
  });
  
  const histories: Record<string, ChatMessageHistory> = {};
  
  function getHistory(sessionId: string): ChatMessageHistory {
    if (!histories[sessionId]) histories[sessionId] = new ChatMessageHistory();
    return histories[sessionId];
  }
  
  const chainWithHistory = new RunnableWithMessageHistory({
    runnable: conversationalRetrievalChain,
    getMessageHistory: getHistory,
    inputMessagesKey: "question",
    historyMessagesKey: "history",
  }).pipe(httpResponseOutputParser);
  
  /* -------------------- Step 4: HTTP handler -------------------- */
  const handler = async (req: Request): Promise<Response> => {
    if (req.method === "POST") {
      try {
        const body = await req.json();
  
        // Pull the right session history and invoke chain
        const answer = await chainWithHistory.invoke(
          {
            question: body.question,
            session_id: body.session_id,
            history: await getHistory(body.session_id).getMessages(),
          },
          { configurable: { sessionId: body.session_id } },
        );
  
        return new Response(JSON.stringify({ reply: answer }), {
          headers: { "content-type": "application/json" },
        });
      } catch (err) {
        console.error("❌ Server error:", err);
        return new Response("Internal Server Error", { status: 500 });
      }
    }
  
    return new Response("✅ Server is running", { status: 200 });
  };
  
  /* -------------------- Step 5: Start server -------------------- */
  Deno.serve({ port: 8087, hostname: "0.0.0.0" }, handler);

  const port = 8087;

const response = await fetch(`http://localhost:${port}`, {
  method: "POST",
  headers: { "content-type": "application/json" },
  body: JSON.stringify({ question: "Tell me my rights", session_id: "1" }),
});

const data = await response.json();

// decode here
const charCodes = Object.values(data.reply) as number[];
const decodedReply = new TextDecoder("utf-8").decode(new Uint8Array(charCodes));

console.log(decodedReply); // ✅ works

  